# 04 — Submission File Generation

This notebook loads all trained models and the processed test dataset to generate 
submission files for each model.

The following submission files will be produced:

- Logistic Regression  
- Balanced Logistic Regression  
- Linear SVM  
- RBF SVM  
- Fast RBF SVM (RFF)  
- Tuned RBF SVM  
- Sigmoid SVM  
- Neural Network (Baseline)  
- Neural Network (LBFGS Stable)  
- Neural Network (SMOTE + Threshold Tuning)  
- FAISS Classifier  

All CSV files will be saved in:  
**`../submissions/submissions_20/`**


In [5]:
import pandas as pd
import numpy as np
import joblib
import os
import faiss

# Create submissions directory
os.makedirs("../submissions/submissions_20", exist_ok=True)

In [6]:
# Load processed train & test
train = pd.read_csv("../data/processed/processed_20/processed_train_20.csv")
test = pd.read_csv("../data/processed/processed_20/processed_test_20.csv")

# Load raw test for ProfileID
raw_test = pd.read_csv("../data/raw/test_updated.csv")

# Add ProfileID column back
test["ProfileID"] = raw_test["ProfileID"]

# Prepare final matrix
X_test = test.drop("ProfileID", axis=1)

print("Train shape:", train.shape)
print("Test shape:", X_test.shape)
test.head()


Train shape: (40856, 25)
Test shape: (51070, 24)


,ApplicantYears,AnnualEarnings,RequestedSum,TrustMetric,WorkDuration,ActiveAccounts,OfferRate,RepayPeriod,DebtFactor,OwnsProperty,...,WorkCategory_Part-time,WorkCategory_Self-employed,WorkCategory_Unemployed,RelationshipStatus_Married,RelationshipStatus_Single,FundUseCase_Business,FundUseCase_Education,FundUseCase_Home,FundUseCase_Other,ProfileID
0,55,112656,92393,581,113,2,23.54,36,0.15,1,...,False,True,False,False,True,False,False,True,False,CKV34LU7V7
1,56,91569,131575,641,54,1,15.19,12,0.43,1,...,True,False,False,False,False,False,True,False,False,62KTYNH93J
2,26,78169,75417,569,105,3,18.02,12,0.29,1,...,True,False,False,True,False,False,True,False,False,JGFUSOIUH7
3,26,63033,10804,326,118,1,14.71,24,0.41,0,...,True,False,False,False,True,True,False,False,False,4538THBHOX
4,24,29665,21182,662,102,3,15.02,60,0.69,0,...,False,False,True,False,True,True,False,False,False,DXLNA06JHR


In [ ]:
## ------------------------------
# Load all saved models
# ------------------------------
log_reg            = joblib.load("../models/models_20/logistic.pkl")
log_reg_bal        = joblib.load("../models/models_20/logistic_balanced.pkl")
rbf_svm            = joblib.load("../models/models_20/rbf_svm.pkl")
fast_rbf_svm       = joblib.load("../models/models_20/fast_rbf_svm.pkl")
sigmoid_svm        = joblib.load("../models/models_20/sigmoid_svm.pkl")
nn1                = joblib.load("../models/models_20/nn_baseline.pkl")
nn_stable          = joblib.load("../models/models_20/nn_lbfgs.pkl")
nn_smote           = joblib.load("../models/models_20/nn_smote.pkl")
faiss_info         = joblib.load("../models/models_20/faiss.pkl")  # dict

# ------------------------------
# Helper to save CSV
# ------------------------------
def save_preds(name, preds):
    df_sub = pd.DataFrame({
        "ProfileID": test["ProfileID"],
        "RiskFlag": preds
    })
    df_sub.to_csv(f"../submissions/submissions_20/{name}.csv", index=False)
    print(f"Saved {name}.csv")

# ------------------------------
# Logistic Regression
# ------------------------------
save_preds("log_reg_submission",      log_reg.predict(X_test))
save_preds("log_reg_balanced_submission", log_reg_bal.predict(X_test))

# ------------------------------
# SVM Models
# ------------------------------
save_preds("rbf_svm_submission",      rbf_svm.predict(X_test))
save_preds("fast_rbf_svm_submission", fast_rbf_svm.predict(X_test))
save_preds("sigmoid_svm_submission",  sigmoid_svm.predict(X_test))

# ------------------------------
# Baseline Neural Network
# ------------------------------
save_preds("nn_baseline_submission", nn1.predict(X_test))
save_preds("nn_lbfgs_submission",    nn_stable.predict(X_test))

# ------------------------------
# NN SMOTE + Threshold Tuning
# ------------------------------
probs = nn_smote.predict_proba(X_test)[:, 1]
t = faiss_info.get("nn_smote_threshold", 0.5)
save_preds("nn_smote_submission", (probs >= t).astype(int))

# ------------------------------
# FAISS MODEL (SPECIAL CASE)
# ------------------------------
index = faiss_info["index"]
pca = faiss_info["pca"]
threshold = faiss_info["threshold"]
y_train_arr = faiss_info["y_train"]
w0, w1 = faiss_info["class_weights"]

X_test_np  = np.ascontiguousarray(X_test.values.astype(np.float32))
X_test_pca = np.ascontiguousarray(pca.transform(X_test_np))
faiss.normalize_L2(X_test_pca)

def faiss_predict_test(K=20):
    distances, neighbors = index.search(X_test_pca, K)
    preds = []

    for dist_list, idx_list in zip(distances, neighbors):
        sim = 1 - dist_list / 2
        sim = np.maximum(sim, 1e-6)

        score0 = np.sum(sim[y_train_arr[idx_list] == 0] * w0)
        score1 = np.sum(sim[y_train_arr[idx_list] == 1] * w1)

        p1 = score1 / (score1 + score0 + 1e-9)
        preds.append(1 if p1 >= threshold else 0)

    return np.array(preds)

faiss_preds = faiss_predict_test()
save_preds("faiss_submission", faiss_preds)

print("\n ALL SUBMISSION FILES GENERATED SUCCESSFULLY!")
print("Check ../submissions/ folder.")

Saved log_reg_submission.csv
Saved log_reg_balanced_submission.csv
